In [35]:
# 1-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_1.csv')
# print(df.info())
# print(df.head())

df['신고시각']=df['신고시각'].astype(str)
df['신고시각']=df['신고시각'].str.zfill(6)
df['처리시각']=df['처리시각'].astype(str)
df['처리시각']=df['처리시각'].str.zfill(6)
df['신고일자']=df['신고일자'].astype(str)
df['처리일자']=df['처리일자'].astype(str)

df['신고일시']=df['신고일자']+df['신고시각']
df['처리일시']=df['처리일자']+df['처리시각']
df['신고일시']=pd.to_datetime(df['신고일시'], format='%Y%m%d%H%M%S', errors='coerce')
df['처리일시']=pd.to_datetime(df['처리일시'], format='%Y%m%d%H%M%S', errors='coerce')
df['rs']=(df['처리일시'] - df['신고일시']).dt.total_seconds()
# print(help(df['rs'].dt.total_seconds)

# print(df.info())
# print(df.head())

df.groupby('공장명')['rs'].mean().sort_values()

공장명
공장B   -1.647024e+06
공장C   -1.598311e+06
공장D    3.107579e+05
공장A    9.021685e+05
공장E    9.247514e+05
Name: rs, dtype: float64

In [7]:
# 1-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_2.csv')
# print(df.info())
# print(df.head())

df['구']=df['STATION_ADDR1'].str.split(' ').str[1]
# print(df.info())
# print(df.head())
print(np.round(df.groupby('구')['dist'].mean(),2).loc[['마포구', '성동구']])

구
마포구    3045.88
성동구    3202.31
Name: dist, dtype: float64


In [47]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_1_3.csv')
# print(df.info())
# print(df.head())
df['year']=df['기간'].str.split('_').str[0].str.replace('년', '').astype(int)
df['month']=df['기간'].str.split('_').str[1].str.replace('월', '').astype(int)
df['sum']=df[['제품A', '제품B', '제품C', '제품D', '제품E']].sum(axis=1)
# print(df.info())
# print(df.head())
def check(x):
    if x in [1,2,3]:
        return 1
    elif x in [4,5,6]:
        return 2
    elif x in [7,8,9]:
        return 3
    elif x in [10,11,12]:
        return 4
    else:
        return np.nan
df['분기']=df['month'].apply(check)
# print(df.info())
# print(df.head())

(df.groupby(['year', '분기'])['sum'].sum()/3).sort_values(ascending=False)

year  분기
2018  3     3226.666667
2019  3     3118.333333
2018  1     3064.000000
      4     2959.000000
2019  1     2914.000000
      4     2456.000000
2018  2     2270.000000
2019  2     1812.333333
Name: sum, dtype: float64

In [138]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

train_y=train_df['Target']
train_X=train_df.drop(columns=['level_0', 'ID', 'Target'])
test_y=test_df['Target']
test_X=test_df.drop(columns=['level_0', 'ID', 'Target'])
train_y=train_y.map({'Dropout': 2, 'Enrolled': 0, 'Graduate': 1})
test_y=test_y.map({'Dropout': 2, 'Enrolled': 0, 'Graduate': 1})
# print(train_X, train_y, test_X, test_y)
# print(train_X.info())
# print(test_X.info())

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import f1_score, get_scorer_names
from sklearn.ensemble import RandomForestClassifier

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()
# print(obj,num)
def check(x):
    return x.str.replace('_', '').astype(int)

train_X[obj]=train_X[obj].apply(check)
test_X[obj]=test_X[obj].apply(check)

obj=train_X.select_dtypes('object').columns.tolist()
num=train_X.select_dtypes('number').columns.tolist()
# print(obj,num)

# print(train_X.info())
# print(test_X.info())

# train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3, stratify=train_y)
train_X, valid_X, train_y, valid_y=train_test_split(train_X, train_y, test_size=0.3)
# print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

# rf=RandomForestClassifier(class_weight='balanced')
rf=RandomForestClassifier()
rf.fit(train_X, train_y)
pred1=rf.predict(valid_X)
print(f1_score(valid_y, pred1, average='macro'))

params={
    'max_depth': [10,20,30],
    'min_samples_split': [2,5,10]
}
cv=GridSearchCV(estimator=rf, param_grid=params, scoring='f1_macro', cv=3)
cv.fit(train_X, train_y)
pred2=cv.predict(valid_X)
print(f1_score(valid_y, pred2, average='macro'))

pred3=rf.predict(test_X)
print(f1_score(test_y, pred3, average='macro'))

pred4=cv.predict(test_X)
print(f1_score(test_y, pred4, average='macro'))

train_X_full=np.concatenate([train_X, valid_X],axis=0)
train_y_full=np.concatenate([train_y, valid_y],axis=0)

rf.fit(train_X_full, train_y_full)
pred5=rf.predict(test_X)
print(f1_score(test_y, pred5, average='macro'))

cv.fit(train_X_full, train_y_full)
pred6=cv.predict(test_X)
print(f1_score(test_y, pred6, average='macro'))

pd.DataFrame(pred6, columns=['pred']).to_csv('result.csv', index=False)

0.5311890838206628
0.5618833118833119
0.5752283945402542
0.5826519272386969
0.5590014543222724


C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


0.5774762815981979


C:\Users\user\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(


In [51]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_1.csv')
# print(df.info())
# print(df.head())

print(np.round(df['before'].mean(),2))
print(np.round(df['before'].std(),2))
print(np.round(df['after'].mean(),2))
print(np.round(df['after'].std(),2))
from scipy.stats import ttest_rel
print(np.round(ttest_rel(df['before'], df['after']).statistic,2))
print(ttest_rel(df['before'], df['after']).pvalue < 0.05)

71.41
11.37
76.3
11.94
-7.9
True


In [69]:
# 3-2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/7_3_2_test.csv')
# print(train_df.info())
# print(train_df.head())
# print(test_df.info())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model=smf.glm('purchase ~ age + income + family_members', data=train_df, family=sm.families.Binomial()).fit()
# print(model.summary())
print(np.round(np.exp(model.params['income']),3))
# print(dir(model))
print(np.round(model.deviance,2))

test_y=test_df['purchase']
test_X=test_df.drop(columns=['purchase'])
pred1=model.predict(test_X)
pred1=(pred1 >= 0.5).astype(int)
from sklearn.metrics import accuracy_score
print(1-accuracy_score(test_y, pred1))

1.0
108.19
0.35
